# M3L4 E00 — Logs tradicionales vs Tracing estructurado
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

**Objetivo:** entender por qué los logs tradicionales no alcanzan para debuggear sistemas de agentes y qué ventajas ofrece el tracing estructurado.

## Conceptos
- **Log tradicional:** registra eventos sueltos, sin jerarquía ni contexto.
- **Trace:** registra el recorrido completo de una request, incluyendo orden, jerarquía, inputs, outputs, errores y duración por paso.
- **Span:** un paso lógico dentro de una trace.

## Parte 1 — Logs tradicionales

Así luce el logging típico que todos conocemos.

In [ ]:
def process_request_with_logs(query: str):
    print('INFO: request received')
    print(f'INFO: user query = {query}')
    print('INFO: classified intent')
    print('INFO: agent executed')
    print('INFO: response returned')

process_request_with_logs('No puedo ver mi factura')

## 🤔 Preguntas para reflexionar

Analizá la salida de la celda anterior y respondé:

1. ¿Se puede saber **qué intent** detectó el sistema?
2. ¿Se puede saber **qué agente** respondió?
3. ¿Se puede saber **cuánto tardó** cada paso?
4. ¿Se puede saber si hubo **retrieval vacío**?
5. ¿Se puede saber si hubo **un error** en algún paso intermedio?

> **Tu respuesta aquí:** (doble click para editar)

## Parte 2 — Trace estructurado

Ahora mirá cómo se vería la misma request como traza estructurada.

El siguiente diccionario representa la **misma request** que los logs de arriba — pero con toda la información organizada.

In [ ]:
trace = {
    'trace_name': 'support-request',
    'input': 'No puedo ver mi factura',
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': 'No puedo ver mi factura',
            'output': {'intent': 'finance'},
            'duration_ms': 120
        },
        {
            'name': 'finance-agent',
            'input': 'No puedo ver mi factura',
            'output': {'answer': 'Podés ver tu factura desde el portal de pagos.'},
            'duration_ms': 860
        }
    ],
    'output': {
        'final_answer': 'Podés ver tu factura desde el portal de pagos.'
    }
}

trace

## 🧩 TODO — Completar la comparación

Comparando logs vs trace, completá la siguiente tabla en Markdown:

| Pregunta | ¿Logs lo responden? | ¿Trace lo responde? |
|---|---|---|
| ¿Qué intent se detectó? | TODO | TODO |
| ¿Qué agente respondió? | TODO | TODO |
| ¿Cuánto tardó cada paso? | TODO | TODO |
| ¿Hubo retrieval vacío? | TODO | TODO |
| ¿Dónde ocurrió el error? | TODO | TODO |

## Parte 3 — Trace con error

Este trace muestra una **misclassification**: el sistema enrutó mal la consulta.

In [ ]:
bad_trace = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {
        'expected_intent': 'finance',
        'actual_intent': 'it'
    },
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'intent': 'it'},
            'duration_ms': 140
        },
        {
            'name': 'it-agent',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'response': 'Probá reiniciar la app.'},
            'duration_ms': 900
        }
    ],
    'output': {
        'final_response': 'Probá reiniciar la app.'
    }
}

bad_trace

## 🧩 TODO — Analizar el error

1. ¿En qué span ocurrió el problema?
2. ¿Cuál era el intent esperado y cuál fue el real?
3. ¿Qué información del trace permite detectar el bug?
4. ¿Qué cambio harías para corregirlo?

> **Tu respuesta aquí:**

## ✅ Síntesis

| Logs tradicionales | Tracing estructurado |
|---|---|
| Eventos sueltos, sin contexto | Historia completa de la request |
| No muestran jerarquía | Muestra spans en orden |
| Sin input/output por paso | Input y output en cada span |
| Difícil de medir latencias | `duration_ms` por span |
| Difícil de detectar misclassifications | `metadata.expected_intent` vs `actual_intent` |

**En el próximo ejercicio (E01)** vas a construir tu propio sistema de tracing en Python puro — así cuando uses Langfuse, vas a entender exactamente qué hace internamente.